# Day 2 : Tokenizer & Generation Internals

## Import & setup


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

## 2. Tokenization Deep Dive

In [15]:
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
texts = ['AI','Artificial intelligence','Ai egineer build systems']
for text in texts:
  tokens = tokenizer.tokenize(text)
  ids = tokenizer.encode(text)
  print(f"text:'{text}'")
  print(f"  Word count : {len(text.split())}")
  print(f"  Token count: {len(tokens)}")
  print(f"  Tokens     : {tokens}")
  print(f"  IDs        : {ids}")
  print()

text:'AI'
  Word count : 1
  Token count: 1
  Tokens     : ['AI']
  IDs        : [20185]

text:'Artificial intelligence'
  Word count : 2
  Token count: 3
  Tokens     : ['Art', 'ificial', 'Ġintelligence']
  IDs        : [8001, 9542, 4430]

text:'Ai egineer build systems'
  Word count : 4
  Token count: 7
  Tokens     : ['A', 'i', 'Ġeg', 'ine', 'er', 'Ġbuild', 'Ġsystems']
  IDs        : [32, 72, 29206, 500, 263, 1382, 3341]



### Observation

- "AI" is 1 word, 1 token
- "Artificial Intelligence" is 2 words but tokenizes into more tokens
- Token count is NOT the same as word count
- BPE splits rare/long words into subword units
- The `Ġ` prefix means "preceded by a space"

## 3. Forward Pass & Logits

In [16]:
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

prompt = "AI engineers build systems"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print("Logits shape:", logits.shape)
print(f"  batch_size      : {logits.shape[0]}")
print(f"  sequence_length : {logits.shape[1]}")
print(f"  vocab_size      : {logits.shape[2]}")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Logits shape: torch.Size([1, 4, 50257])
  batch_size      : 1
  sequence_length : 4
  vocab_size      : 50257


### What does logits shape mean?
- `batch_size` = 1 → we passed one prompt
- `sequence_length` = number of tokens in the prompt
- `vocab_size` = 50257 → one score for every possible next token

For each position in the sequence, the model outputs a score
for every word in its vocabulary. Higher score = more likely next token.

## 4. Manual Next Token Prediction

In [17]:
# Get the last token's logits — this predicts what comes NEXT
last_token_logits = logits[0, -1, :]
print("Last token logits shape:", last_token_logits.shape)

# Greedy: pick the highest scoring token
next_token_id = torch.argmax(last_token_logits).item()
next_token = tokenizer.decode([next_token_id])

print(f"\nPredicted next token ID : {next_token_id}")
print(f"Predicted next token    : '{next_token}'")

Last token logits shape: torch.Size([50257])

Predicted next token ID : 326
Predicted next token    : ' that'


In [18]:
# Verify: greedy generate() should predict the same first new token
gen_output = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=1,
    do_sample=False  # greedy
)

# The last token in the output is the generated one
generated_token_id = gen_output[0, -1].item()
generated_token = tokenizer.decode([generated_token_id])

print(f"model.generate() next token : '{generated_token}'")
print(f"Manual argmax next token    : '{next_token}'")
print(f"Match: {next_token == generated_token}")

model.generate() next token : ' that'
Manual argmax next token    : ' that'
Match: True


### 5. Chat Templates (Chapter 3 Integration)
Base models like distilgpt2 just complete text.
Instruct models expect a specific input format — the chat template.

## 5. Chat Templates (Chapter 3 integration)

In [19]:
from transformers import AutoTokenizer

tokenizer_instruct = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is tokenization?"}
]

formatted = tokenizer_instruct.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is tokenization?<|im_end|>
<|im_start|>assistant



### What do we see?
- Special tokens like `<|im_start|>`, `<|user|>`, `<|assistant|>` appear
- These are not natural language — they are control tokens
- The model was trained to recognize these and respond accordingly
- distilgpt2 was never trained with these tokens → chat template won't work on it

## 6. Observations & Notes



### Why does word count != token count?
BPE (Byte Pair Encoding) splits text into subword units based on
frequency in training data. Common short words = 1 token.
Rare or long words = multiple tokens.

### What does logits[0, -1, :] mean?
- `0` = first (only) item in the batch
- `-1` = last token position in the sequence
- `:` = all vocab scores
We use `-1` because the last position predicts what comes NEXT.

### What is the difference between manual argmax and model.generate()?
Manual argmax does ONE step — predicts one next token.
model.generate() loops: predict → append → predict → append...
until max_new_tokens or EOS token is reached.

### Why can't distilgpt2 use chat templates?
distilgpt2 is a base model trained only on next-token prediction.
It was never trained on instruction-following data or with special
role tokens like <|user|> or <|assistant|>. Applying a chat template
would just confuse it — the special tokens would be treated as
unknown text.